# 04 · Relational Persona Graph-RAG (P-Graph)
**Framework II of LPS — Graph-RAG-based Persona Modeling**

Implements:
1. **Triplet extraction** from each interaction: $(s, p, o, \kappa)$
2. **Temporally decayed edge weights**:

$$W(e_{uv}) = \sum_{t \in \mathcal{T}_{uv}} \gamma^{(t_{\text{now}} - \tau_t)}$$

3. **Leiden hierarchical community detection** → Persona Sub-graphs
4. **Multi-tier relational retrieval** (Global → Structural → Exemplar)


In [ ]:

import sys
sys.path.append("../src")

import numpy as np
import json, yaml, pickle
import networkx as nx
import matplotlib.pyplot as plt
import anthropic
from graph_rag import PersonaGraphRAG

with open("../configs/default.yaml") as f:
    cfg = yaml.safe_load(f)
with open("../data/cohort.json") as f:
    cohort = json.load(f)
with open("../data/embeddings.pkl", "rb") as f:
    all_embeddings = pickle.load(f)

uid = list(cohort.keys())[0]
data = all_embeddings[uid]
client = anthropic.Anthropic()
print(f"User: {uid[:12]}... — {len(data['interactions'])} interactions")


In [ ]:

# ── Extract triplets from first 10 interactions (demo) ────────────────
pgraph = PersonaGraphRAG(
    gamma_decay=cfg["graph_rag"]["gamma_decay"],
    leiden_resolution=cfg["graph_rag"]["leiden_resolution"],
    hop_depth=cfg["graph_rag"]["hop_depth"],
    anthropic_client=client,
)

all_triplets = []
interactions = data["interactions"]
timestamps   = data["timestamps"]

print("Extracting triplets (first 10 interactions)...")
for i, (inter, ts) in enumerate(zip(interactions[:10], timestamps[:10])):
    triplets = pgraph.extract_triplets(inter["user"], ts)
    all_triplets.extend(triplets)
    if triplets:
        print(f"  [{i}] '{inter['user'][:50]}...'")
        for t in triplets[:2]:
            print(f"        ({t.subject}, {t.predicate}, {t.obj}) [κ={t.kappa}]")


In [ ]:

# ── Build knowledge graph ─────────────────────────────────────────────
if all_triplets:
    t_now = max(t.timestamp for t in all_triplets)
    pgraph.build_graph(all_triplets, t_now=t_now)
    G = pgraph.graph_

    print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
    print(f"\nTop 5 edges by weight:")
    edges_by_weight = sorted(G.edges(data=True),
                              key=lambda x: x[2]["weight"], reverse=True)
    for u, v, d in edges_by_weight[:5]:
        print(f"  ({u}) --[{d['predicates'][0]}]--> ({v})  W={d['weight']:.3f}")
else:
    print("No triplets extracted — check API key and connection")


In [ ]:

# ── Visualise knowledge graph ─────────────────────────────────────────
if pgraph.graph_.number_of_nodes() > 0:
    G = pgraph.graph_
    fig, ax = plt.subplots(figsize=(12, 8))

    pos = nx.spring_layout(G, k=2, seed=42)
    weights = [G[u][v]["weight"] for u, v in G.edges()]
    max_w   = max(weights) if weights else 1

    nx.draw_networkx_nodes(G, pos, node_size=300,
                           node_color="steelblue", alpha=0.8, ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=7, ax=ax)
    nx.draw_networkx_edges(G, pos,
                           width=[2 * w / max_w for w in weights],
                           alpha=0.6, edge_color="gray",
                           arrows=True, arrowsize=10, ax=ax)

    ax.set_title(f"Persona Knowledge Graph — User {uid[:12]}...
"
                 f"Edge thickness ∝ temporal weight W(e_uv)", fontsize=12)
    ax.axis("off")
    plt.tight_layout()
    plt.savefig("../data/persona_graph.png", dpi=120)
    plt.show()


In [ ]:

# ── Tier 1 retrieval ─────────────────────────────────────────────────
from embeddings import UserPrioritizedEmbedder

embedder  = UserPrioritizedEmbedder(alpha=cfg["embedding"]["alpha"])
test_conv = list(cohort[uid]["test"])[0]
test_query = test_conv["pairs"][0]["user"]
query_emb  = embedder.encode([test_query])[0]

# Build a simple node embeddings map for community detection demo
node_emb_map = {
    node: embedder.encode([node])[0]
    for node in list(pgraph.graph_.nodes())[:20]
}

try:
    pgraph.detect_communities(node_emb_map)
    print(f"Communities detected: {len(pgraph.communities_)}")
    for c in pgraph.communities_:
        print(f"  Community {c.community_id}: {len(c.nodes)} nodes — {c.nodes[:3]}")
except ImportError as e:
    print(f"Leiden not available: {e}")
    print("Install with: pip install igraph leidenalg")

result = pgraph.retrieve(query_emb)
print(f"\nRetrieval complete:")
print(f"  Top communities : {len(result['community_prior'])}")
print(f"  Subgraph edges  : {len(result['subgraph_edges'])}")
print(f"  Exemplar triplets: {len(result['exemplar_triplets'])}")
print("\n✓ Notebook 04 complete — Graph-RAG framework ready")
